# Medica: Dataset Pipeline on Kaggle

Index the uploaded figures, verify caption to image matches, then preprocess. Attach both datasets before running.

In [ ]:
# --- Setup: get the repo into the notebook runtime -----------------------
# On Kaggle: add the repo as a "utility" input dataset, or set REPO_URL.
# The clone lands in /kaggle/working, which is the only writable area and
# what gets saved with the version.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/NickLitharis/medica.git"
WORK = Path("/kaggle/working")
REPO = WORK / "medica"

if not REPO.exists():
    try:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
    except Exception as e:
        print("git clone failed:", e)
        print("Upload the repo as a Kaggle dataset and set REPO to that folder instead.")
    else:
        print("Cloned to", REPO)

if REPO.exists():
    sys.path.insert(0, str(REPO))
    os.chdir(REPO)
    print("Working directory:", os.getcwd())
else:
    print("Repo missing: the rest of this notebook cannot run.")


In [ ]:
# --- Resolve input and output paths --------------------------------------
# Attach the datasets first (right sidebar -> Add Data):
#   * figure images  (PNG files, folder name does not matter)
#   * s2_cleaned.json (cleaned captions JSONL)
from config.paths import resolve, check
paths = resolve()
check(paths)


In [ ]:
# --- Stage 1: index the images and verify the caption matches -------------
!python3 scripts/create_image_index.py
!python3 scripts/fix_and_create_verified_index.py


In [ ]:
# --- Stage 2: preprocess images to 512x512 -------------------------------
# Writes into /kaggle/working/medica/preprocessed/images. Kaggle gives a
# notebook about 20 GB of working disk, so 105k images at 512px will not fit.
# Run this stage with --limit for a subset, or reduce the size to 384/224.
!python3 scripts/preprocess_images.py --limit 20000


In [ ]:
# --- Stage 3: ViT features (cached as one .npy per figure) ----------------
# Optional: the attention captioner reads the images directly, so it does not
# need these files. Use this stage only for the classical encoder path or to
# cache features for a lighter notebook.
!pip install -q transformers
!python3 scripts/extract_features.py
